Готовим данные и обучаем модель для `app.py`. После выполнения всех ячеек рядом появятся `model.joblib` и `clients.csv`.


Клиенты оператора связи, одна строка – один клиент. `churn` – ушёл ли клиент.


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/Murcha1990/datasets/refs/heads/main/churn_cliens_demo.csv"
df = pd.read_csv(url)
print(f"Размер: {df.shape}")  # 3333 клиента, 20 колонок
print(f"Пропусков: {df.isnull().sum().sum()}")  # заполнять нечего


Размер: (3333, 20)
Пропусков: 0


Признаки, которые возьмём в модель, и доля ушедших.


In [2]:
FEATURES = [
    "accountlength",
    "internationalplan",
    "voicemailplan",
    "totaldayminutes",
    "customerservicecalls",
]
print(df[FEATURES + ["churn"]].head(3))
print(df["churn"].value_counts(normalize=True).round(3))


   accountlength internationalplan voicemailplan  totaldayminutes  customerservicecalls  churn
0            128                No           Yes            265.1                     1  False
1            107                No           Yes            161.6                     1  False
2            137                No            No            243.4                     0  False
churn
False    0.855
True     0.145


Перевод `Yes` / `No` и масштабирование кладём внутрь `Pipeline`. Приложение потом отдаст модели те же колонки, что в CSV.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

YES_NO = ["internationalplan", "voicemailplan"]

X = df[FEATURES]
y = df["churn"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

yes_no = ColumnTransformer(
    [("yes_no", OrdinalEncoder(categories=[["No", "Yes"]] * len(YES_NO)), YES_NO)],
    remainder="passthrough",  # остальные колонки – без изменений
)
model = make_pipeline(yes_no, StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]  # вероятность класса 1, «ушёл»
print(f"Accuracy: {accuracy_score(y_test, model.predict(X_test)):.3f}")
print(f"ROC-AUC на тесте: {roc_auc_score(y_test, y_pred_proba):.3f}")


Accuracy: 0.859
ROC-AUC на тесте: 0.805


Сохраняем модель и 20 случайных клиентов без колонки `churn` – их загрузим в приложение.


In [4]:
from pathlib import Path

import joblib

joblib.dump(model, "model.joblib")
print(f"Модель: model.joblib, {Path('model.joblib').stat().st_size} байт")

sample = df.drop(columns=["churn"]).sample(20, random_state=42)
sample.to_csv("clients.csv", index=False)
print(f"Файл для загрузки: clients.csv, {len(sample)} строк, без churn")


Модель: model.joblib, 3905 байт
Файл для загрузки: clients.csv, 20 строк, без churn
